In [ ]:
from pathlib import Path
import re
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
HERE = Path.cwd().resolve()
ROOT = HERE.parent
OUT = HERE / 'outputs' / '08_enrichment'
TAB = OUT / 'tables'
FIG = OUT / 'figures'
TAB.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
DONORS = ['P1', 'P2', 'Q1', 'Q2', 'S1', 'S2']
AMINO = list('ACDEFGHIKLMNPQRSTVWY')
AA_RE = re.compile('^[ACDEFGHIKLMNPQRSTVWY]+$')
YFV_PAT = re.compile('yellow\\s*fever|\\byfv\\b|17d|yellow\\s*fever\\s*virus', re.I)
CMV_PAT = re.compile('cytomegalovirus|\\bcmv\\b|hhv-?5|human\\s*herpesvirus\\s*5|human\\s*cytomegalovirus', re.I)
FISHER_TAB = ROOT / 'results' / 'fisher_day0_day15' / 'tables'
FISHER_LOCAL = HERE / 'outputs' / '02_fisher' / 'tables'
POGO_LOCAL = HERE / 'outputs' / '03_pogorelyy' / 'tables'
POGO_TAB = ROOT / 'pogorelyy_repro' / 'results' / 'tables'
VDJDB = ROOT / 'results' / 'fisher_day0_day15_extended' / 'databases' / 'vdjdb-2026-06-03' / 'vdjdb.slim.txt'
YF_CSV = ROOT / 'results' / 'day0_day15_external_tcr_annotation' / 'databases' / 'Yellow_Fever_TCRs.csv'
MOTIF = ROOT / 'results' / 'day0_day15_external_tcr_annotation' / 'databases' / 'tcr_motif_atlas_full.csv'


In [ ]:
def norm_aa(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ''
    s = str(x).strip().upper().replace(' ', '')
    return s if AA_RE.match(s) and len(s) >= 5 else ''

def expand_radius(seeds, radius):
    cur = set(seeds)
    for _ in range(radius):
        nxt = set(cur)
        for s in cur:
            chars = list(s)
            for (i, c) in enumerate(chars):
                for a in AMINO:
                    if a == c:
                        continue
                    chars[i] = a
                    nxt.add(''.join(chars))
                chars[i] = c
        cur = nxt
    return cur

def contingency_enrichment(n_mark_hit, n_mark, n_bg_hit, n_bg):
    table = [[n_mark_hit, n_mark - n_mark_hit], [n_bg_hit, n_bg - n_bg_hit]]
    (oddsr, p) = fisher_exact(table, alternative='greater')
    (a, b, c, d) = (table[0][0], table[0][1], table[1][0], table[1][1])
    (aa, bb, cc, dd) = (a + 0.5, b + 0.5, c + 0.5, d + 0.5)
    log_or = np.log(aa * dd / (bb * cc))
    se = np.sqrt(1 / aa + 1 / bb + 1 / cc + 1 / dd)
    (lo, hi) = (np.exp(log_or - 1.96 * se), np.exp(log_or + 1.96 * se))
    return {'marker_hit_n': int(n_mark_hit), 'background_hit_n': int(n_bg_hit), 'marker_hit_pct': 100.0 * n_mark_hit / n_mark if n_mark else np.nan, 'background_hit_pct': 100.0 * n_bg_hit / n_bg if n_bg else np.nan, 'odds_ratio': float(oddsr) if np.isfinite(oddsr) else np.nan, 'or_ci95_low': float(lo), 'or_ci95_high': float(hi), 'p_value': float(p)}


In [ ]:
ref_rows = []
vdj = pd.read_csv(VDJDB, sep='\t', low_memory=False)
vdj = vdj[vdj['gene'].astype(str).str.upper().isin(['TRB', 'TCRB', 'BETA'])].copy()
vdj['cdr3_aa'] = vdj['cdr3'].map(norm_aa)
vdj = vdj[vdj['cdr3_aa'] != '']
blob = vdj['antigen.species'].astype(str) + ' ' + vdj.get('antigen.gene', pd.Series([''] * len(vdj))).astype(str) + ' ' + vdj.get('antigen.epitope', pd.Series([''] * len(vdj))).astype(str)
vdj['pathogen'] = np.select([blob.str.contains(YFV_PAT, na=False), blob.str.contains(CMV_PAT, na=False)], ['YFV', 'CMV'], default='OTHER')
for patho in ['YFV', 'CMV']:
    sub = vdj[vdj['pathogen'] == patho]
    for aa in sub['cdr3_aa']:
        ref_rows.append({'source': 'VDJdb', 'pathogen': patho, 'cdr3_aa': aa})
if YF_CSV.exists():
    yf = pd.read_csv(YF_CSV)
    col = next((c for c in yf.columns if c.lower() in ('cdr3', 'cdr3_aa', 'cdr3aa', 'cdr3_trb')), None)
    if col:
        for aa in yf[col].map(norm_aa):
            if aa and aa.startswith(('CAS', 'CS', 'CAT', 'CAI', 'CAW', 'CAR')):
                ref_rows.append({'source': 'YF_CSV', 'pathogen': 'YFV', 'cdr3_aa': aa})
if MOTIF.exists():
    mot = pd.read_csv(MOTIF, low_memory=False, usecols=lambda c: c in ('cdr3_TRB', 'source_organism_common', 'source_protein', 'peptide'))
    if 'cdr3_TRB' in mot.columns:
        mot = mot[mot['cdr3_TRB'].notna()].copy()
        mot['cdr3_aa'] = mot['cdr3_TRB'].map(norm_aa)
        mot = mot[mot['cdr3_aa'] != '']
        blob = mot.get('source_organism_common', pd.Series([''] * len(mot))).astype(str) + ' ' + mot.get('source_protein', pd.Series([''] * len(mot))).astype(str) + ' ' + mot.get('peptide', pd.Series([''] * len(mot))).astype(str)
        mot['pathogen'] = np.select([blob.str.contains(YFV_PAT, na=False), blob.str.contains(CMV_PAT, na=False)], ['YFV', 'CMV'], default='OTHER')
        for patho in ['YFV', 'CMV']:
            for aa in mot.loc[mot['pathogen'] == patho, 'cdr3_aa']:
                ref_rows.append({'source': 'MotifAtlas', 'pathogen': patho, 'cdr3_aa': aa})
refs = pd.DataFrame(ref_rows).drop_duplicates()
yfv_exact = set(refs.loc[refs.pathogen == 'YFV', 'cdr3_aa'])
cmv_exact = set(refs.loc[refs.pathogen == 'CMV', 'cdr3_aa'])
yfv_ham1 = expand_radius(yfv_exact, 1)
cmv_ham1 = expand_radius(cmv_exact, 1)
refs.to_csv(TAB / 'reference_records.csv', index=False)


In [ ]:
def load_fisher_donor(donor):
    for base in (FISHER_LOCAL, FISHER_TAB):
        p = base / f'fisher_all_clonotypes_{donor}.parquet'
        if p.exists():
            return pd.read_parquet(p, columns=['cdr3_aa', 'cdr3_nt', 'fisher_positive_FC4', 'q_value', 'fold_change'])
        p2 = base / f'fisher_positive_FC4_{donor}.csv'
        if p2.exists() and (base / f'fisher_all_clonotypes_{donor}.parquet').exists() is False:
            pass
    p = FISHER_TAB / f'fisher_all_clonotypes_{donor}.parquet'
    return pd.read_parquet(p, columns=['cdr3_aa', 'cdr3_nt', 'fisher_positive_FC4', 'q_value', 'fold_change'])

def load_model_aa(donor):
    for base in (POGO_LOCAL, POGO_TAB):
        for name in (f'model_positive_{donor}.csv', f'{donor}_15_top_expanded.tsv'):
            p = base / name
            if p.exists():
                df = pd.read_csv(p, sep='\t' if p.suffix == '.tsv' else ',')
                return set(df['cdr3_aa'].map(norm_aa)) - {''}
    return set()
ann_parts = []
for donor in DONORS:
    df = load_fisher_donor(donor)
    df['cdr3_aa'] = df['cdr3_aa'].map(norm_aa)
    df = df[df['cdr3_aa'] != '']
    g = df.groupby('cdr3_aa', as_index=False).agg(marker_fisher=('fisher_positive_FC4', 'max'), n_nt=('cdr3_nt', 'nunique'), min_q=('q_value', 'min'))
    model_aa = load_model_aa(donor)
    g['marker_pogorelyy'] = g['cdr3_aa'].isin(model_aa)
    g['donor'] = donor
    g['yfv_exact'] = g['cdr3_aa'].isin(yfv_exact)
    g['yfv_ham1'] = g['cdr3_aa'].isin(yfv_ham1)
    g['cmv_exact'] = g['cdr3_aa'].isin(cmv_exact)
    g['cmv_ham1'] = g['cdr3_aa'].isin(cmv_ham1)
    ann_parts.append(g)
ann = pd.concat(ann_parts, ignore_index=True)
ann.to_parquet(TAB / 'annotated_unique_cdr3aa.parquet', index=False)
ann.head()


In [ ]:
def donor_enrichment(ann, marker_col, hit_col, label):
    rows = []
    for donor in DONORS:
        d = ann[ann['donor'] == donor]
        mark = d[d[marker_col]]
        bg = d[~d[marker_col]]
        stats = contingency_enrichment(int(mark[hit_col].sum()), len(mark), int(bg[hit_col].sum()), len(bg))
        rows.append({'donor': donor, 'marker_set': label, 'hit_definition': hit_col, 'n_markers': len(mark), 'n_background': len(bg), **stats})
    out = pd.DataFrame(rows)
    (_, q, _, _) = multipletests(out['p_value'], method='fdr_bh')
    out['q_value'] = q
    return out
tests = [('marker_fisher', 'yfv_exact', 'Fisher / YFV exact'), ('marker_fisher', 'yfv_ham1', 'Fisher / YFV ham1'), ('marker_fisher', 'cmv_exact', 'Fisher / CMV exact'), ('marker_pogorelyy', 'yfv_exact', 'Pogorelyy / YFV exact'), ('marker_pogorelyy', 'yfv_ham1', 'Pogorelyy / YFV ham1'), ('marker_pogorelyy', 'cmv_exact', 'Pogorelyy / CMV exact')]
all_enr = [donor_enrichment(ann, m, h, lab) for (m, h, lab) in tests]
enrichment = pd.concat(all_enr, ignore_index=True)
enrichment.to_csv(TAB / 'enrichment_by_donor.csv', index=False)
enrichment.head(12)


In [ ]:
plot = enrichment[enrichment['marker_set'].isin(['Fisher / YFV exact', 'Pogorelyy / YFV exact', 'Fisher / CMV exact'])]
(fig, axes) = plt.subplots(1, 2, figsize=(11, 4))
for (ax, metric, title) in zip(axes, ['marker_hit_pct', 'odds_ratio'], ['% annotated among markers', 'Odds ratio vs same-donor background']):
    for (label, color) in [('Fisher / YFV exact', '#C0392B'), ('Pogorelyy / YFV exact', '#2980B9'), ('Fisher / CMV exact', '#7f8c8d')]:
        sub = plot[plot['marker_set'] == label]
        ax.plot(sub['donor'], sub[metric], marker='o', label=label, color=color)

    ax.legend(fontsize=7)
    if metric == 'odds_ratio':
        ax.set_yscale('log')
plt.tight_layout()
plt.savefig(FIG / 'yfv_enrichment.png', dpi=150, bbox_inches='tight')
plt.show()
rows = []
for (label, g) in enrichment.groupby('marker_set'):
    rows.append({'marker_set': label, 'mean_marker_pct': g['marker_hit_pct'].mean(), 'mean_bg_pct': g['background_hit_pct'].mean(), 'median_OR': g['odds_ratio'].median(), 'n_donors_q005': int((g['q_value'] < 0.05).sum())})
pooled = pd.DataFrame(rows)
pooled.to_csv(TAB / 'enrichment_summary.csv', index=False)
pooled


In [ ]:
EPI = ROOT / "results" / "yfv_enrichment" / "tables" / "cluster_epitope_enrichment.tsv"
if EPI.exists():
    epi = pd.read_csv(EPI, sep="\t")
    epi.to_csv(TAB / "cluster_epitope_enrichment.csv", index=False)
    top = epi.sort_values("q_value").head(15) if "q_value" in epi.columns else epi.head(15)
    top
